# Flocking: Vicsek model and boids

*others · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/boids_flocking)

Starlings, fish schools and locusts move together without a leader. In the Vicsek model each bird flies at constant speed and copies the average heading of its neighbours, plus a random angle of size up to $\eta$; in Reynolds' boids it steers by separation, alignment and cohesion. All $N^2$ bird-to-bird distances in the periodic box form one tensor, and a whole batch of noise levels runs at once to show the order–disorder transition of the polar order $\varphi$.

**Vector form**

$$\theta_i \leftarrow \arg\Big(\sum_{|\mathbf d_{ij}| < R} e^{i\theta_j}\Big) + \eta\,\xi_i,\qquad \mathbf x_i \leftarrow \mathbf x_i + v_0\,\hat{\mathbf e}(\theta_i),\qquad \varphi = \frac{\lvert\sum_i \mathbf v_i\rvert}{\sum_i \lvert\mathbf v_i\rvert}$$

**Index (tensor) form**

$$d_{bijk} = x_{bjk} - x_{bik} - L\,\operatorname{round}\!\big(\tfrac{x_{bjk} - x_{bik}}{L}\big),\qquad A_{bij} = [\,d_{bijk}d_{bijk} < R^2\,],\qquad \theta_{bi} \leftarrow \arg\!\big(A_{bij}\,e^{i\theta_{bj}}\big) + \eta_b\,\xi_{bi}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bik}$ | `random_flock(B, N, L, device, seed)` | (B, N, 2) | positions of $N$ birds in the box $[0, L)^2$, one flock per noise level $b$ |
| $d_{bijk}$ | `minimum_image(x[:, None] - x[:, :, None], L)` | (B, N, N, 2) | displacement from bird $i$ to the nearest periodic copy of bird $j$; antisymmetric in $i \leftrightarrow j$ |
| $A_{bij}$ | `(d.pow(2).sum(-1) < R**2).float()` | (B, N, N) | neighbour matrix, repeated $k$ ⇒ summed; bird $i$ counts itself |
| $\arg(A_{bij} e^{i\theta_{bj}})$ | `einsum('bij,bj->bi', A, exp(1j * theta)).angle()` | (B, N) | repeated $j$ ⇒ summed: mean heading of the neighbours |
| $\eta_b\,\xi_{bi}$ | `eta * (torch.rand(B, N) - 0.5)` | (B, N) | noise angle, uniform in $[-\eta/2, \eta/2]$; $\eta = 2\pi$ makes the heading completely random |
| $\mathbf a_i$ | `reynolds_acceleration(x, v, L, R, weights)` | (B, N, 2) | boids: separation $-\sum_j S_{ij} \mathbf d_{ij}//\mathbf d_{ij}/^2$, alignment $\sum_j A_{ij}(\mathbf v_j - \mathbf v_i)$, cohesion $\sum_j A_{ij}\mathbf d_{ij}$ |
| $\varphi_b$ | `v.sum(1).norm(dim=-1) / v.norm(dim=-1).sum(1)` | (B,) | polar order: 1 when the flock moves together, about $1/\sqrt N$ when headings are random |

Vicsek et al. (1995) introduced the model as the simplest physics of collective motion: alignment plus noise. At low noise the birds agree on a direction ($\varphi \to 1$), at $\eta = 2\pi$ the headings are random and $\varphi$ is only finite-size noise of order $1/\sqrt N$. In two dimensions this is remarkable, because the Mermin–Wagner theorem forbids long-range order in equilibrium systems with a continuous symmetry; moving birds escape it because neighbourhoods keep changing (Toner and Tu 1995). For large systems the transition turns out to be discontinuous, with travelling bands (Grégoire and Chaté 2004); at this small $N$ it looks smooth. Reynolds (1987) wrote boids for computer animation. Here his three rules are written as pair forces that are antisymmetric in $i \leftrightarrow j$, so without the speed normalisation the total momentum of the flock is conserved exactly (a test checks it); the normalisation to constant speed $v_0$ then lets the flock pick a direction. Strong cohesion makes the flock collapse into a rotating ball, so it is kept weak. Try a smaller box (higher density) or a larger radius to make order easier.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/boids_flocking/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import polar_order


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    n = 80 if fast else params["n_birds"]
    box, radius, v0 = params["box"], 1.0, params["speed"]
    n_steps = 40 if fast else params["steps"]
    eta = torch.cat([tensors.noise_levels(6 if fast else params["n_noise"], dev), torch.tensor([[params["eta"]]])])

    x, theta = tensors.random_flock(len(eta), n, box, dev, params["seed"])  # (B, N, 2), (B, N)
    xs, thetas, phi = solve.vicsek(x, theta, eta, n_steps, box, radius, v0, params["seed"] + 1)
    late = phi[:, n_steps // 2 :].mean(1)  # (B,) time average after the transient
    print(f"Vicsek: N = {n} birds in a {box}×{box} periodic box (density {n / box**2:.1f}), R = {radius}, v0 = {v0}")
    for e, f in zip(eta[:-1, 0].tolist(), late[:-1].tolist()):
        print(f"  η = {e:4.2f}: polar order φ = {f:.3f}")
    print(f"  maximal noise η = 2π: φ = {late[-2]:.3f}  (finite-size level 1/√N = {1 / math.sqrt(n):.3f})")
    step = (xs[1:] - xs[:-1] - box * torch.round((xs[1:] - xs[:-1]) / box)).norm(dim=-1)
    print(
        f"  speed: min {step.min():.5f}, max {step.max():.5f} (constant v0); all birds in the box: "
        f"{bool(((xs >= 0) & (xs < box)).all())}"
    )

    xb, vb = solve.boids(
        x[-1:], tensors.velocities(theta[-1:], v0), n_steps, 1.0, box, radius, tensors.reynolds_weights(v0), v0
    )
    phi_b = polar_order(vb.flatten(0, 1))  # (T + 1,)
    print(f"Reynolds boids (separation, alignment, cohesion): φ from {phi_b[0]:.3f} to {phi_b[-1]:.3f}")

    shown = {"eta": params["eta"], "x": xs[:, -1].cpu(), "theta": thetas[:, -1].cpu(), "phi": phi[-1].cpu()}
    boid = {"x": xb[:, 0].cpu(), "v": vb[:, 0].cpu(), "phi": phi_b.cpu()}
    return plot.animate(shown, boid, eta[:-1, 0].cpu(), phi[:-1].cpu(), late[:-1].cpu(), box, out_dir / "flock.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/boids_flocking/equation.py
r"""Flocking rules → tensor translation.

Every bird interacts with the birds within a radius R in a periodic box of side L. All N² displacements are one
tensor, folded back with the minimum-image convention (the nearest copy of bird j):

    \mathbf d_{ij} = (\mathbf x_j - \mathbf x_i) - L\,\operatorname{round}\!\big((\mathbf x_j - \mathbf x_i)/L\big),
    \qquad A_{ij} = [\,|\mathbf d_{ij}| < R\,].

Vicsek model (Vicsek et al. 1995). Each bird flies at constant speed v_0 and takes the average heading of its
neighbours (itself included) plus a random angle in [−η/2, η/2]:

    \theta_i \leftarrow \arg\Big(\sum_j A_{ij}\,e^{i\theta_j}\Big) + \eta\,\xi_i,\qquad
    \mathbf x_i \leftarrow \mathbf x_i + v_0\,(\cos\theta_i, \sin\theta_i).

Boids (Reynolds 1987): separation, alignment and cohesion, here written as pairwise forces

    \mathbf a_i = \sum_j \Big[-w_s S_{ij}\frac{\mathbf d_{ij}}{|\mathbf d_{ij}|^2}
                  + w_a A_{ij}(\mathbf v_j - \mathbf v_i) + w_c A_{ij}\,\mathbf d_{ij}\Big],

each antisymmetric in i ↔ j, so Σ_i a_i = 0 (Newton's third law) until the speed is renormalised to v_0.
The polar order parameter φ = |Σ_i v_i| / Σ_i |v_i| is 1 for a flock moving together and ~1/√N for disorder.
"""

import torch


def minimum_image(d: torch.Tensor, box: float) -> torch.Tensor:
    """Fold a displacement into [−L/2, L/2] in each direction: the nearest periodic copy.   (...) → (...)"""
    return d - box * torch.round(d / box)


def pair_displacements(x: torch.Tensor, box: float) -> torch.Tensor:
    """d_ij = x_j − x_i with the minimum image.   (B, N, 2) → (B, N, N, 2)"""
    return minimum_image(x[:, None, :, :] - x[:, :, None, :], box)


def neighbours(d: torch.Tensor, radius: float) -> torch.Tensor:
    """A_ij = 1 if |d_ij| < R (bird i itself included).   (B, N, N, 2) → (B, N, N) float"""
    return (d.pow(2).sum(-1) < radius**2).to(d.dtype)


def polar_order(v: torch.Tensor) -> torch.Tensor:
    """φ = |Σ_i v_i| / Σ_i |v_i|   (i summed).   (B, N, 2) → (B,)"""
    return v.sum(1).norm(dim=-1) / v.norm(dim=-1).sum(1)


def vicsek_headings(x: torch.Tensor, theta: torch.Tensor, box: float, radius: float) -> torch.Tensor:
    r"""Mean heading of the neighbours: arg Σ_j A_ij e^{iθ_j}   (j summed).   (B, N, 2), (B, N) → (B, N)"""
    a = neighbours(pair_displacements(x, box), radius)  # (B, N, N)
    z = torch.einsum("bij,bj->bi", a.to(torch.complex64), torch.exp(1j * theta).to(torch.complex64))
    return z.angle().to(theta.dtype)


def vicsek_step(
    x: torch.Tensor, theta: torch.Tensor, eta: torch.Tensor, xi: torch.Tensor, box: float, radius: float, v0: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """One Vicsek update for every noise level at once: η (B, 1), ξ uniform in [−½, ½] (B, N).
    (B, N, 2), (B, N) → new positions (B, N, 2) in [0, L), new headings (B, N)"""
    theta = vicsek_headings(x, theta, box, radius) + eta * xi
    step = v0 * torch.stack([torch.cos(theta), torch.sin(theta)], -1)  # (B, N, 2) constant speed v0
    return torch.remainder(x + step, box), theta


def reynolds_acceleration(x: torch.Tensor, v: torch.Tensor, box: float, radius: float, w: dict) -> torch.Tensor:
    """Separation (within R/4), alignment and cohesion (within R) as antisymmetric pair forces.
    (B, N, 2), (B, N, 2) → (B, N, 2)"""
    d = pair_displacements(x, box)  # (B, N, N, 2)
    a = neighbours(d, radius)  # (B, N, N)
    r2 = d.pow(2).sum(-1)
    close = neighbours(d, radius / 4) * (r2 > 0)  # bird i itself excluded
    repel = torch.einsum("bij,bijk->bik", close / r2.clamp(min=1e-12), d)  # Σ_j S_ij d_ij / |d_ij|²
    align = torch.einsum("bij,bjk->bik", a, v) - a.sum(-1, keepdim=True) * v  # Σ_j A_ij (v_j − v_i)
    cohere = torch.einsum("bij,bijk->bik", a, d)  # Σ_j A_ij d_ij: pull towards the local centre
    return -w["separation"] * repel + w["alignment"] * align + w["cohesion"] * cohere


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/boids_flocking/tensors.py
"""Build the concrete tensors: a random flock in a periodic box, the noise levels and the Reynolds weights."""

import math

import torch

# Reynolds weights per unit time for v0 = 0.05 (neighbour radius R = 1, about 20 neighbours at the default density).
# Strong cohesion makes the flock collapse into a milling ball, so cohesion is kept weak.
REYNOLDS = {"separation": 0.0003, "alignment": 0.02, "cohesion": 0.0001}


def reynolds_weights(speed: float) -> dict:
    """Separation and cohesion steer relative to the speed, so they scale with v0 / 0.05."""
    return {k: w * (speed / 0.05 if k != "alignment" else 1.0) for k, w in REYNOLDS.items()}


def random_flock(batch: int, n: int, box: float, device: torch.device, seed: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Uniform positions in [0, L)² and uniform headings, independent for every batch member:
    positions (B, N, 2), headings θ (B, N)."""
    gen = torch.Generator().manual_seed(seed)
    x = box * torch.rand(batch, n, 2, generator=gen)
    theta = 2 * math.pi * torch.rand(batch, n, generator=gen) - math.pi
    return x.to(device), theta.to(device)


def noise_levels(count: int, device: torch.device) -> torch.Tensor:
    """count noise amplitudes η from 0 to 2π (2π = completely random headings): (B, 1)."""
    return torch.linspace(0, 2 * math.pi, count, device=device)[:, None]


def velocities(theta: torch.Tensor, speed: float) -> torch.Tensor:
    """v = v0 (cos θ, sin θ).   (B, N) → (B, N, 2)"""
    return speed * torch.stack([torch.cos(theta), torch.sin(theta)], -1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/boids_flocking/solve.py
"""Time stepping: the Vicsek model for a batch of noise levels, and Reynolds boids."""

import torch

from problems._common.progress import steps

from .equation import polar_order, reynolds_acceleration, vicsek_step


def vicsek(
    x: torch.Tensor,
    theta: torch.Tensor,
    eta: torch.Tensor,
    n_steps: int,
    box: float,
    radius: float,
    v0: float,
    seed: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Run every noise level η_b (B, 1) at once from x (B, N, 2), θ (B, N).
    → positions (T + 1, B, N, 2), headings (T + 1, B, N), polar order φ (B, T + 1)."""
    gen = torch.Generator().manual_seed(seed)
    xs, thetas = [x], [theta]
    for _ in steps(n_steps, "Vicsek steps"):
        xi = (torch.rand(theta.shape, generator=gen) - 0.5).to(theta.device)  # uniform in [−½, ½]
        x, theta = vicsek_step(x, theta, eta, xi, box, radius, v0)
        xs.append(x)
        thetas.append(theta)
    thetas = torch.stack(thetas)
    unit = torch.stack([torch.cos(thetas), torch.sin(thetas)], -1)  # (T + 1, B, N, 2)
    phi = polar_order(unit.flatten(0, 1)).reshape(len(thetas), -1).T  # (B, T + 1)
    return torch.stack(xs), thetas, phi


def boids(
    x: torch.Tensor,
    v: torch.Tensor,
    n_steps: int,
    dt: float,
    box: float,
    radius: float,
    weights: dict,
    speed: float | None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Semi-implicit Euler: v ← v + a dt (then |v| ← v0 if speed is given), x ← x + v dt in the periodic box.
    (B, N, 2), (B, N, 2) → positions (T + 1, B, N, 2), velocities (T + 1, B, N, 2)."""
    xs, vs = [x], [v]
    for _ in steps(n_steps, "boids steps"):
        v = v + dt * reynolds_acceleration(x, v, box, radius, weights)
        if speed is not None:
            v = speed * v / v.norm(dim=-1, keepdim=True).clamp(min=1e-12)
        x = torch.remainder(x + dt * v, box)
        xs.append(x)
        vs.append(v)
    return torch.stack(xs), torch.stack(vs)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/boids_flocking/plot.py
"""A flock of arrows forming (Vicsek and boids) beside the order parameter φ against the noise η."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(
    shown: dict, boid: dict, eta: torch.Tensor, phi: torch.Tensor, late: torch.Tensor, box: float, path: Path
) -> Path:
    """shown: Vicsek run at one noise (x (T, N, 2), theta (T, N), phi (T,)); boid: x, v (T, N, 2), phi (T,);
    eta (B,), phi (B, T), late (B,) for the batch of noise levels."""
    xv, th = shown["x"].numpy(), shown["theta"].numpy()
    xb, vb = boid["x"].numpy(), boid["v"].numpy()
    ub = vb / np.linalg.norm(vb, axis=-1, keepdims=True)
    e, p, avg = eta.numpy(), phi.numpy(), late.numpy()
    n_t = len(th)

    fig = plt.figure(figsize=(11, 3.9))
    grid = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.25], wspace=0.28)
    panels = []
    for g, name in ((grid[0], f"Vicsek, η = {shown['eta']:.2f}"), (grid[1], "Reynolds boids")):
        ax = fig.add_subplot(g)
        ax.set_xlim(0, box)
        ax.set_ylim(0, box)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        panels.append((ax, ax.set_title(name, fontsize=10), name))
    q_v = panels[0][0].quiver(
        xv[0, :, 0],
        xv[0, :, 1],
        np.cos(th[0]),
        np.sin(th[0]),
        th[0],
        cmap="twilight",
        clim=(-np.pi, np.pi),
        scale=18,
        width=0.007,
    )
    q_b = panels[1][0].quiver(
        xb[0, :, 0], xb[0, :, 1], ub[0, :, 0], ub[0, :, 1], color="#2563eb", scale=18, width=0.006
    )

    ax_p = fig.add_subplot(grid[2])
    ax_p.plot(e, avg, color="#d1d5db", lw=2, label="time average (end)")
    (now,) = ax_p.plot(e, p[:, 0], "o-", color="#dc2626", ms=4, lw=1.2, label="φ now, every η at once")
    ax_p.axvline(shown["eta"], color="#6b7280", ls=":", lw=1)
    ax_p.axhline(1 / np.sqrt(xv.shape[1]), color="#f59e0b", ls="--", lw=1, label="1/√N (no order)")
    ax_p.set_xlim(0, 2 * np.pi)
    ax_p.set_ylim(0, 1.03)
    ax_p.set_xlabel("noise η")
    ax_p.set_ylabel("polar order φ = |Σ v| / Σ |v|")
    ax_p.set_title("order–disorder transition", fontsize=10)
    ax_p.legend(fontsize=7, frameon=False, loc="upper right")
    clock = ax_p.text(0.03, 0.05, "", transform=ax_p.transAxes, fontsize=8, color="#6b7280")

    def update(t: int):
        q_v.set_offsets(xv[t])
        q_v.set_UVC(np.cos(th[t]), np.sin(th[t]), th[t])
        q_b.set_offsets(xb[t])
        q_b.set_UVC(ub[t, :, 0], ub[t, :, 1])
        panels[0][1].set_text(f"{panels[0][2]}: φ = {shown['phi'][t]:.2f}")
        panels[1][1].set_text(f"{panels[1][2]}: φ = {boid['phi'][t]:.2f}")
        now.set_ydata(p[:, t])
        clock.set_text(f"step {t}")

    return save_gif(fig, update, frame_indices(n_t, 50), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_birds': 200,
    'box': 5.8,
    'speed': 0.03,
    'eta': 1.5,
    'n_noise': 12,
    'steps': 500,
    'seed': 0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.boids_flocking")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.boids_flocking.main")
problem_dir = Path("problems/others/boids_flocking")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
